# Build Product Models #

This JupyterNotebook file will use the selected based model to produce the product models. 

**Functions:**

1 - modifyModels(inputDirectory_createModels, outputDirectory_createModels)

2 - addProductionPW(model, files)

**Requirements:** needs updating!!!
- Models to be edited must be.xml files and have the prefix "PRE" added to the file name (ie. PREmodel_GS_Farnesene.xml); the PRE prefix will be removed in the exported files. Code could be modified to change this requirement if desired.
- Excel file format be used for adding new pathways/reactions to the model(s).
- When designing the pathways in the Excel file formate, users should ensure that the reaction and metabolite IDs being used are consistent with the BiGG database (the code will check IDs against the model to ensure nothing is added in duplicate, but this relies on correct naming convention on the user side).

** *See SPI Analysis Log at bottom of document*

In [1]:
import cobra
import cameo
import math
import escher
import plotly
import os

import numpy as np
from scipy import stats
import pandas as pd
import sympy as sy
from datetime import date, datetime
import time
import glob #For createModels() function

date = datetime.strftime(datetime.now(), '%Y-%m-%d')

In [67]:
#EH Note from 2023-10-09: I think this code block was for a specific purpose; I don't think it's actually needed for running this code.
modelName = 'iHN637_Propane-PW2.xml'
model = cameo.load_model('SP_Investigation/Overall_Analysis_Docs/CreateProductModels/Input/' + modelName)
#model =  cameo.models.bigg.iHN637 
model.reactions.get_by_id('ADO2mp').lower_bound = 0
model.reactions.get_by_id('ADO2mp').upper_bound = 0

model.reactions.get_by_id('ALDD2mpc3').upper_bound = 1000
model.reactions.get_by_id('EX_fru_e').lower_bound = 0
cobra.io.write_sbml_model(model, 'SP_Investigation/Overall_Analysis_Docs/CreateProductModels/Output/' + 'new_' + modelName)

In [23]:
#If a model other than iJO1366 or iHN637 is used as base, must save it as an xml file first:
model =  cameo.models.bigg.iHN637 #iJN1463
modelName = 'iHN637_Glutamate_L.xml'#'model_iJN1463.xml'
cobra.io.write_sbml_model(model, 'SP_Investigation/Overall_Analysis_Docs/CreateProductModels/Input/' + modelName)

In [5]:
## Main - Modifying models ##
#Run other functions below this cell first

#Directory for models to be edited:
inputDirectory_createModels = os.path.abspath("CreateProductModels/Input/")

#Directory for model output after editing:
outputDirectory_createModels = os.path.abspath("CreateProductModels/Output/")

#ID for base model
modelID = 'iHN637' #Currently set up to accept 'iHN637', 'iJO1366' OR name of downloaded xml file (ie. 'modelName.xml') located in Input folder

tic = time.perf_counter() #Timer start
#Add the pathways to the models and export them to desired folder:

modifyModels(inputDirectory_createModels, outputDirectory_createModels, modelID)
toc = time.perf_counter()
print(f"Models modified in {toc - tic:0.4f} seconds")

C:\Users\austi\AppData\Local\Temp\ipykernel_15624\1466023054.py:3: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if modelID is 'iHN637':
C:\Users\austi\AppData\Local\Temp\ipykernel_15624\1466023054.py:6: SyntaxWarning: "is" with a literal. Did you mean "=="?
  elif modelID is 'iJO1366':
C:\Users\austi\AppData\Local\Temp\ipykernel_15624\1466023054.py:22: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if modelID is 'iHN637': modelName = 'iHN637_' + modelName #Indicate iHN637 used as base model
C:\Users\austi\AppData\Local\Temp\ipykernel_15624\1466023054.py:3: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if modelID is 'iHN637':
C:\Users\austi\AppData\Local\Temp\ipykernel_15624\1466023054.py:6: SyntaxWarning: "is" with a literal. Did you mean "=="?
  elif modelID is 'iJO1366':
C:\Users\austi\AppData\Local\Temp\ipykernel_15624\1466023054.py:22: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if modelID is 'iHN637': modelName = 'iHN637_' + modelName 

KeyboardInterrupt: 

In [5]:
#Test code:
modelName = 'TestModel.xml'
model = cameo.load_model(outputDirectory_createModels + modelName)
display(model.reactions.get_by_id('EX_eg_e'))
display(model.reactions.get_by_id('ACPS'))

In [2]:
def modifyModels(inputDirectory_createModels, outputDirectory_createModels, modelID):
    
    if modelID is 'iHN637':
        model = cameo.models.bigg.iHN637
        print("test")
    elif modelID is 'iJO1366':
        model = cameo.models.bigg.iJO1366
    else:
        model = cameo.load_model(inputDirectory_createModels + modelID)
        print('Model other than iJO1366 or iHN637 loaded; may need to ensure necessary supporting reactions added.')
    
    #Add supporting reactions for base model
    model = modifyBaseModel(model, modelID)
    
    for files in glob.glob1(inputDirectory_createModels, '*.xlsx'):
        #Load model
        #model =  cameo.models.bigg.iHN637
        #model =  cameo.models.bigg.iJO1366
        modelName = files.replace(inputDirectory_createModels, '') #Remove folder path from filenames
        modelName = modelName.replace('.xlsx', '.xml')
        
        if modelID is 'iHN637': modelName = 'iHN637_' + modelName #Indicate iHN637 used as base model
        
        print("model name:")
        display(modelName)#Check
        
        #Add production pathway to the model
        modelUpdated = addProductionPW(model.copy(), files) #Pass copy of model such that base model isn't changed for next loop iteration
        print('Pathway added to model:', modelName) #Check
        #display(modelUpdated) #Check
        #display(model) #Check
    
        #Save modified model as a new model:
        cobra.io.write_sbml_model(modelUpdated, outputDirectory_createModels + '\\' + modelName)
        
        print('Model exported:', modelName) #Check
        print('\n')
        
        
        ##---Quick FBA Test---##
        #modelUpdated.objective = 'EX_r13bdo_e'
        
        #modelUpdated.reactions.get_by_id('EX_glc__D_e').lower_bound = -10
        #modelUpdated.reactions.get_by_id('EX_fru_e').lower_bound = 0
        #modelUpdated.reactions.get_by_id('EX_co_e').lower_bound = -20
        #modelUpdated.reactions.get_by_id('EX_h2_e').lower_bound = -40
        
        #fba_result = cameo.fba(modelUpdated)
        #display(modelUpdated.summary())
    
    return()

<>:3: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:6: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:22: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:3: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:6: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:22: SyntaxWarning: "is" with a literal. Did you mean "=="?
C:\Users\austi\AppData\Local\Temp\ipykernel_15624\1466023054.py:3: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if modelID is 'iHN637':
C:\Users\austi\AppData\Local\Temp\ipykernel_15624\1466023054.py:6: SyntaxWarning: "is" with a literal. Did you mean "=="?
  elif modelID is 'iJO1366':
C:\Users\austi\AppData\Local\Temp\ipykernel_15624\1466023054.py:22: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if modelID is 'iHN637': modelName = 'iHN637_' + modelName #Indicate iHN637 used as base model


In [3]:
def modifyBaseModel(model, modelID):
    
    if modelID is 'iHN637':
        
        #iHN637 model uses fructose as default carbon source, turn this off:
        model.reactions.get_by_id('EX_fru_e').lower_bound = 0
        
        #Must allow flow of H+ between periplasm and extra-cellular space
        H_p = cobra.Metabolite('h_p', formula = 'H', name = 'H+', compartment = 'p')
        H_e = model.metabolites.get_by_id('h_e')
        
        reaction = cobra.Reaction('Htex')
        reaction.name = 'Proton transport via diffusion (extracellular to periplasm)'
        reaction.subsystem = "unknown"
        reaction.lower_bound = -1000
        reaction.upper_bound = 1000

        reaction.add_metabolites({H_e: -1, H_p: 1})
        model.add_reactions([reaction])
        
        ##Note:
        #iHN637 model doesn't have periplasm, but product models do for compatibility with iJO1366 model
        #Adding periplasmic space to iHN637 model for production pathways will not change modeling predictions
    
    #Other reactions to add (for quick tests)
    add = False
    if add:
        ##Acetyl-CoA exchange:
        accoa_c = model.metabolites.get_by_id('accoa_c')
        accoa_e = cobra.Metabolite('accoa_e', formula = 'C23H34N7O17P3S', name = 'acetyl-CoA', compartment = 'e')

        #Transport
        reaction = cobra.Reaction('AcCoAtce')
        reaction.name = 'AcCoA transport'
        reaction.subsystem = "unknown"
        reaction.lower_bound = -1000 #export rate
        reaction.upper_bound = 1000 #uptake rate

        reaction.add_metabolites({accoa_e: -1, accoa_c:1})
        model.add_reactions([reaction])

        #Exchange rxn
        reaction = cobra.Reaction('EX_accoa_e')
        reaction.name = 'AcCoa exchange'
        reaction.subsystem = "unknown"
        reaction.lower_bound = 0 #uptake rate (specifies highest rate at which EG enters system)
        reaction.upper_bound = 1000 #export rate (specifies highest rate at which EG can leave the system; ie. be produced)

        reaction.add_metabolites({accoa_e: -1})
        model.add_reactions([reaction])
        
    return(model)

<>:3: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:3: SyntaxWarning: "is" with a literal. Did you mean "=="?
C:\Users\austi\AppData\Local\Temp\ipykernel_15624\2646484595.py:3: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if modelID is 'iHN637':


In [4]:
def addProductionPW(model, files):
    
    #Make lists of all the metabolites in the model and all of the reactions in the model
    met_ids = [metabolite.id for metabolite in model.metabolites]
    rxn_ids = [reaction.id for reaction in model.reactions]
    
    
    #print(files) #Check
    fileNames = files.replace(inputDirectory_createModels, '')
    print('\n',fileNames) #Check
        
    #Read in metabolites
    metabolitesDF = pd.read_excel(inputDirectory_createModels + '\\' + fileNames, sheet_name='Metabolites')
    #display(metabolitesDF) #Check
        
    #Read in reactions
    reactionsDF = pd.read_excel(inputDirectory_createModels + '\\' + fileNames, sheet_name='Reactions')
    #display(reactionsDF) #Check
        
    #Loop through all metabolites in metabolitesDF, add any missing from the model
    for index, rows in metabolitesDF.iterrows():
    
        metID = metabolitesDF.loc[index].metID #Assign metabolite ID from dataframe
        if not metID in met_ids: #If metabolite doesn't yet exist in the model, create new metabolite (otherwise no action needed)
            #Assign metabolite fields from dataframe:
            metFormula = metabolitesDF.loc[index].metFormula
            metName = metabolitesDF.loc[index].metName
            metCompartment = metabolitesDF.loc[index].metCompartment
            metCharge = int(metabolitesDF.loc[index].metCharge)
            #print('metCharge is type: ', type(metCharge))#Check
            #display(metID, metFormula, metName, metCompartment, metCharge) #Check
    
            #Create metabolite:
            metabolite = cobra.Metabolite(metID, formula = metFormula, name = metName, compartment = metCompartment, charge = metCharge)
                
            #Add metabolite to the model:
            model.add_metabolites([metabolite])
            #print('Metabolite added:', metabolite)#Check
            #display(metabolite) #Check
                
            #Add metabolite ID to the list of metIDs, so that if future assimilation pathways have the same metabolite,
            #it's not added twice (or doesn't give error message)
            met_ids.append(metID)
    
    #display(model.metabolites) #Check
    #display(model.metabolites.get_by_id('test_e'))#Check
    
    #Loop through all reactions in reactionsDF, add any missing from the model
    for index, rows in reactionsDF.iterrows():
    
        rxnID = reactionsDF.loc[index].rxnID #Assign reaction ID from dataframe
        if not rxnID in rxn_ids: #If reaction doesn't yet exist in model, create and add it
            reaction = cobra.Reaction(reactionsDF.loc[index].rxnID)
            reaction.name = reactionsDF.loc[index].rxnName
            reaction.subsystem = reactionsDF.loc[index].rxnSubsystem
            reaction.lower_bound = reactionsDF.loc[index].rxnLB
            reaction.upper_bound = reactionsDF.loc[index].rxnUB

            #Add the reaction to the model (Note: its reaction field will be empty at first):
            model.add_reactions([reaction])
    
            #Add the reaction stoichiometry to the new reaction's reaction field (supplied from reactionsDF)
            rxn = model.reactions.get_by_id(rxnID)
            rxn.reaction = reactionsDF.loc[index].rxnAddMetabolites
                
            #Add reaction ID to the list of rxnIDs, so that if future assimilation pathways have the same reaction,
            #it's not added twice (or doesn't give error message)
            rxn_ids.append(rxnID)
            
            #Need to set reaction bounds again after adding reaction to model, or it defaults knocked out reactions [0,0] to [0,1000]
            model.reactions.get_by_id(rxnID).lower_bound = float(reactionsDF.loc[index].rxnLB)
            model.reactions.get_by_id(rxnID).upper_bound = float(reactionsDF.loc[index].rxnUB)
    
            print('Reaction added:', rxn , ' with bounds: [' 
                  + str(model.reactions.get_by_id(rxnID).lower_bound) 
                  + ',' + str(model.reactions.get_by_id(rxnID).upper_bound) + ']') #Check
            #display(rxn) #Check
            
        elif rxnID in rxn_ids: #If reaction already exists in model, make sure it has correct bounds set
            model.reactions.get_by_id(rxnID).lower_bound = float(reactionsDF.loc[index].rxnLB) #Need to convert to int or loat or converting to sbml model will produce error later
            model.reactions.get_by_id(rxnID).upper_bound = float(reactionsDF.loc[index].rxnUB) #Need to convert to int or float or converting to sbml model will produce error later
            print('Reaction modified:' + rxnID + ' bounds set to [' + str(model.reactions.get_by_id(rxnID).lower_bound) 
                  + ',' + str(model.reactions.get_by_id(rxnID).upper_bound) + ']')
            #display(model.reactions.get_by_id(rxnID)) #Check
            
    return(model)

#model = cameo.models.bigg.e_coli_core
#N_metabolites = len(model.metabolites) #if you want to know how many metabolites are in the model
#N_reactions = len(model.reactions) #if you want to know how many reactions are in the model
#print('This model has ', N_metabolites, ' metabolites and ', N_reactions, ' reactions.')

#inputDirectory_editModels = 'SP_Investigation/Overall_Analysis_Docs/CreateModels/Input/EditModels/'
#modelNew = addAssimilationPWs(model, inputDirectory_editModels)
#N_metabolites = len(modelNew.metabolites) #if you want to know how many metabolites are in the model
#N_reactions = len(modelNew.reactions) #if you want to know how many reactions are in the model
#print('This model has ', N_metabolites, ' metabolites and ', N_reactions, ' reactions.')

#display(model.reactions.get_by_id('EX_eg_e'))
#Old: #vars()[metabolitesDF.loc[index].metDef] = cobra.Metabolite(metID, formula = metFormula, name = metName, compartment = metCompartment)
#Old: #vars()[reactionsDF.loc[index].rxnID] = reaction
#display(model.reactions.get_by_id('ACALDt'))#Check (comparison)

**SPI Analysis Log:**
- EH (2021-3-23): Models from XML_2021-3-23 (modified and corrected) were modified to include the following assimilation pathways: 
    - EG/Glycoladehyde via Glycolate pathway: sEG-Glycolaldehyde_pwGlycolate.xlsx
    - EG/Glycoladehyde via SACA pathway: sEG-Glycolaldehyde_pwSACA.xlsx
    - Formate/Methanol via formolase pathway: sFormate-Methanol_pwFormolase.xlsx
    - Formate/Methanol via Serine cycle: sFormate-Methanol_pwSerineCycle.xlsx,
    - Formate via RGP pathway via Serine: sFormate_pwRGPviaSerine.xlsx
    - Methanol via RuMP pathway: sMethanol_pwRuMP.xlsx
    - The acetate, ethanol, acetaladehyde, xylose, glucose and glycerol pathways already exist in the iJO1366 model.